# CO-SPY Fusion v3 từ checkpoint có sẵn

Notebook này dành cho cấu trúc Kaggle trong ảnh:
- `Co-Spy_v1`: dataset `mscoco2017` và `stable-diffusion-v1-4`
- `Co-Spy_v2`: checkpoint `ckpt/sd-v1_4/{semantic,artifact,npr}`

Notebook chỉ train lại Fusion v3; không train lại ba branch.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

REPO = Path('/kaggle/working/CO-SPY')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/datnguyentrung/CO-SPY.git', str(REPO)], check=True)
os.chdir(REPO)
print('Repository:', REPO)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

# Tự áp dụng bản sửa Fusion v3 nếu GitHub vẫn đang ở commit cũ.
fusion_file = REPO / 'detectors' / 'sd_v1_4' / 'fusion.py'
fusion = fusion_file.read_text(encoding='utf-8')
old_dims = 'feature_dims = [self.sem.fc.in_features, self.art.fc.in_features]'
new_dims = 'feature_dims = [self.sem.fc.in_features + self.sem.fc.out_features, self.art.fc.in_features]'
if old_dims in fusion:
    fusion = fusion.replace(old_dims, new_dims)
old_extract = 'sem_feat = self.sem.clip.encode_image(x_sem)'
new_extract = ('            sem_embedding = self.sem.clip.encode_image(x_sem)\n'
               '            sem_logit = self.sem.fc(sem_embedding)\n'
               '            sem_feat = torch.cat([sem_embedding, sem_logit], dim=1)')
if old_extract in fusion:
    fusion = fusion.replace(old_extract, new_extract)
fusion_file.write_text(fusion, encoding='utf-8')

train_file = REPO / 'train.py'
train = train_file.read_text(encoding='utf-8')
anchor = '            self.cache_key = hashlib.sha256('
schema = '            if self.fusion_head != \"v1\":\n                signature += \"|semantic_feature_schema=clip_plus_trained_logit_v1\"\n'
if 'semantic_feature_schema=clip_plus_trained_logit_v1' not in train:
    train = train.replace(anchor, schema + anchor)
    train_file.write_text(train, encoding='utf-8')

assert 'sem_logit = self.sem.fc(sem_embedding)' in fusion_file.read_text(encoding='utf-8')
assert 'semantic_feature_schema=clip_plus_trained_logit_v1' in train_file.read_text(encoding='utf-8')
print('Fusion v3 patch: OK')

In [ ]:
packages = ['diffusers==0.32.2', 'open-clip-torch==2.31.0', 'timm==1.0.15', 'loguru==0.7.3', 'accelerate==1.0.1', 'ftfy==6.2.3', 'transformers==4.46.3', 'safetensors==0.5.3']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Dependencies installed')

In [ ]:
INPUT = Path('/kaggle/input')
def find_dir(name):
    hits = [p for p in INPUT.rglob(name) if p.is_dir()]
    if not hits: raise FileNotFoundError(f'Không tìm thấy {name} trong /kaggle/input')
    return hits[0]

dataset_candidates = [p for p in INPUT.rglob('*') if p.is_dir() and (p / 'mscoco2017').is_dir() and (p / 'stable-diffusion-v1-4').is_dir()]
if not dataset_candidates: raise FileNotFoundError('Không tìm thấy thư mục dataset chứa mscoco2017 và stable-diffusion-v1-4')
data_root = dataset_candidates[0]
ckpt_root = find_dir('ckpt')
print('Dataset root:', data_root)
print('Checkpoint root:', ckpt_root)

# Dataset được đọc qua symlink để không nhân bản dữ liệu lớn.
data_link = REPO / 'data' / 'train' / 'sd-v1_4'
data_link.mkdir(parents=True, exist_ok=True)
for name in ['mscoco2017', 'stable-diffusion-v1-4']:
    link = data_link / name
    if not link.exists(): link.symlink_to(data_root / name, target_is_directory=True)

# Chỉ copy các checkpoint nhỏ vào nơi train.py yêu cầu.
target_ckpt = REPO / 'ckpt'
if target_ckpt.exists(): shutil.rmtree(target_ckpt)
shutil.copytree(ckpt_root, target_ckpt)
for branch in ['semantic', 'artifact', 'npr']:
    files = list((target_ckpt / 'sd-v1_4' / branch).rglob('checkpoint_best.pth'))
    if not files: raise FileNotFoundError(f'Thiếu checkpoint {branch}/checkpoint_best.pth')
    print(branch, files[0], files[0].stat().st_size / 1024**2, 'MB')

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), 'GPUs:', torch.cuda.device_count())
if not torch.cuda.is_available(): raise RuntimeError('Hãy bật Accelerator = GPU trong Kaggle')
!python -m py_compile detectors/sd_v1_4/fusion.py train.py

In [ ]:
# Fusion v3: cache mới sẽ tự được tạo vì semantic feature schema đã đổi.
!python -u main.py --phase train --mode fusion --train_dataset sd-v1_4 --ckpt /kaggle/working/CO-SPY/ckpt --fusion_head v3 --epochs 5 --early-stopping-patience 2 --artifact_views 1 --num-workers 2 --prefetch-factor 2 --precision auto

In [ ]:
from pathlib import Path
for p in sorted((REPO / 'ckpt' / 'sd-v1_4' / 'fusion_v3').rglob('*')):
    if p.is_file(): print(p, f'{p.stat().st_size / 1024**2:.2f} MB')
print('Hoàn tất. Hãy Save Version để giữ checkpoint và feature-cache.')